<a href="https://colab.research.google.com/github/Teivak/FaceRecognitionProject/blob/main/2_HW_ArcFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArcFace Loss (Additive Angular Margin Loss)

## Теория ArcFace

В случае с обучением на задачу классификации первая подходящая лосс-функция, которая нам приходит в голову — Cross-Entropy. И на ней действительно можно обучать сеть для распознавания лиц. Но за много лет люди придумали более хитрые трюки, которые делают обучение сети для распознавания лиц более эффективным. Одним из лучших подходов считается ArcFace (Additive Angular Margin).


**Как устроен ArcFace**:

Стандартные SoftMax + кросс-энтропия (CE) выглядят так:

$$L_{CE} = \frac{-1}{N}\sum_1^N \frac{e^{W_{y_i}^{T}x_i + b_{y_i}}}{\sum^n_{j=1}e^{W_j^Tx_i+b_j}},$$

здесь:
- $x_i \in \mathbb{R^d}$ — вектор $i$-го элемента обучающей выборки перед последним полносвязным слоем сети. $y_i$ — класс этого элемента;
- $W_j \in \mathbb{R^d}$ — j-ый столбец матрицы весов последнего слоя сети (т.е. слоя, который производит итоговую классификацю входящего объекта);
- $b_j \in \mathbb{R^d}$ — j-ый элемент вектора байеса последнего слоя сети;
- $N$ — batch size;
- $n$ — количество классов.


Хотя этот лосс работает хорошо, он явным образом не заставляет эмбеддинги $x_i$ элементов, принадлежащих одному классу, быть близкими друг к другу по расстоянию. И не заставляет эмбеддинги элементов, принадлежащих разным классам, быть далеко друг от друга. Все, что хочет этот лосс — чтобы на основе эмбеддингов $x_i$ можно было хорошо классифицировать элементы, никакие ограничений на расстояния между эмбеддингами $x_i$ он не вводит.

Из-за этого у нейросетей для распознавания лиц, которые обучены на обычном CE loss, бывают проблемы с распознаванием лиц, которые сильно отличаются от лиц того же человека разными доп. атрибутами (шляпа/прическа/очки и т.п.). Просто эмбеддинг для таких лиц получается довольно далек по расстоянию от других эмбеддингов лиц этого же человека.

Давайте теперь немного поправим формулу:
- уберем байес последнего слоя, т.е. сделаем $b_j=0$;
- нормализуем веса последнего слоя: ||$W_j$|| = 1;
- нормализуем эмбеддинги: ||$x_i$|| = 1. Перед подачей их на вход последнему слою (т.е. перед умножением на матрицу $W_j$) умножим их на гиперпараметр s. По сути, мы приводим норму всех эмбеддингов к s. Смысл этого гиперпараметра в том, что, возможно, сети проще будет классифицировать эмбеддинги, у которых не единичная норма.

Нормализация приводит к тому, что эмбеддинги распределяются по сфере единичного радиуса (и сфере радиуса s после умножения на гиперпараметр s). И итоговые предсказания сети после последнего слоя зависят только от угла между эмбеддингами $x_i$ и выученных весов $W_j$. От нормы эмбеддинга $x_i$ они больше не зависят, т.к. у всех эмбеддингов они теперь одинаковые.

Получается, в степени экспоненты у нас останется выражение $s W_{y_i}^{T}x_i$, которое можно переписать в виде  $s W_{y_i}^{T}x_i = s ||W_{y_i}||\cdot ||x_i|| \cdot cos\Theta_{y_i}$. Тут $\Theta_{y_i}$ — это угол между векторами $W_{y_i}$ и $x_i$. Но так как мы сделали нормы $W_{y_i}$ и $x_i$ единичными, то все это выражение просто будет равно $s cos\Theta_{y_i}$.

В итоге мы получим следующую формулу лосса:

$$L = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos\Theta_{y_i}}}{e^{s\ cos\Theta_{y_i}} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$


И последний шаг. Добавим еще один гиперпараметр $m$. Он называется additive angular margin penalty и заставляет эмбеддинги одного класса быть ближе друг к другу, а эмбеддинги разных классов — более далекими друг от друга.

В итоге получим вот что:

$$L_{ArcFace} = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos(\Theta_{y_i} + m)}}{e^{s\ cos(\Theta_{y_i} + m)} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$

Это и есть ArcFace Loss с двумя  гиперпараметрами, s и m.

Получается, что ArcFace Loss завтавляет сеть выучивать эмбеддинги, распределенные по сфере радиуса s, причем чтобы эмбеддинги одного класса были ближе друг к другу, а эмбеддинги разных классов — более далеки друг от друга.

![ArcFace](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTKRR-YA_XR3yhIYBbkc8Zlbua0Q2WdM3gx_g&s)

**Важное пояснение:**

Строго говоря, ArcFace - не лосс, отдельный архитектурный модуль модификация SoftMax. Он реализует идею внесения геометрического отступа непосредственно в пространство признаков. Для обучения в качестве лосса используется обычная кросс-энтропия. Более конкретно по шагам:

1. Вы извлекаете эмбеддинги из бэкбона сети (предобученной модели, у которой обрезан FC-слой, если он был)
2. Эти эмбеддинги поступают в ArcFace-слой, который содержит векторы-центры для каждого класса (веса классификатора) и логику нормализации и добавления углового отступа
3. Для целевого класса ArcFace-слой преобразует косинус угла $\theta$ в $cos(\theta + m)$
4. Для остальных классов оставляет обычный косинус $cos(\theta)$
5. Эти модифицированные логиты подаются на вход стандартной функции Cross-Entropy
6. Градиенты от Cross-Entropy текут назад через ArcFace-слой к бэкбону, обучая модель извлекать эмбеддинги

Результат: модифицированные логиты с "жестким" разделением для целевого класса, а значит и более качественные эмбеддинги.

Схема:
```
[Изображение] → [Бэкбон] → [ЭМБЕДДИНГ] → [ArcFace] → [Логиты] → [CE Loss]
                    │                        │           │          
                   CNN                   Нормализация   Оценки
                                          + Angular    для всех
                                            Margin     классов
```

Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают. Он нужен был только обучения модели, и поэтому часто ArcFace называю именно лоссом. Но стоит всегда держать в голове, что это некоторое упрощение, которое нужно лишь для того, чтобы проще формулировать мысли.

**Доп. литература по ArcFace Loss:**

Оригинальная статья: https://arxiv.org/pdf/1801.07698.pdf

## Другие лоссы

Кроме ArcFace, есть еще много разных вариантов лоссов для задачи Face Recognition. Некоторые из них можно найти, например, [тут](https://openaccess.thecvf.com/content_CVPRW_2020/papers/w48/Hsu_A_Comprehensive_Study_on_Loss_Functions_for_Cross-Factor_Face_Recognition_CVPRW_2020_paper.pdf). Вы можете попробовать реализовать другие лосс-функции в этом проекте в качестве дополнительного задания.

Кроме этого, можно миксовать лосс-функции. Например, обучать нейросеть на сумме ArcFace и TripletLoss. Иногда так выходит лучше, чем если обучать на каком-то одном лоссе.

# Датасет

В качестве датасета нужно использовать картинки из CelebA, выровненные при помощи своей модели из задания 1. Очень желательно их еще кропнуть таким образом, чтобы нейросети поступали на вход преимущественно только лица без какого либо фона, частей тела и прочего.

Если планируете делать дополнительное задание на Identificaton rate metric, то **обязательно разбейте заранее датасет на train/val или train/val/test.** Это нужно сделать не только на уровне кода, а на уровне папок, чтобы точно знать, на каких картинках модель обучалась, а на каких нет. Лучше заранее почитайте [ноутбук с заданием](https://colab.research.google.com/drive/15zuNdOupRFnG7oE-rFj9FsjoNTK6DYn5).

# План заданий

Итак, вот, что от вас требуется в этом задании:

* Выбрать модель (или несколько моделей) для обучения. Можно брать предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц.
* Обучить эту модель (модели) на CE loss. Добиться accuracy > 0.7.
* Реализовать ArcFace loss.
* Обучить модель (модели) на ArcFace loss. Добиться accuracy > 0.7.
* Написать небольшой отчет по обучению, сравнить CE loss и ArcFace loss.

**P.S. Не забывайте сохранять модели после обучения**

In [1]:
import numpy as np
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from torchvision.tv_tensors import KeyPoints
import cv2 # Импортируем cv2 для функции face_align
from PROJECT.FaceAlignment.face_align import face_align # Добавляем явный импорт face_align

# Мы сохраняем этот блок кода отдельно, так что желательно,
# чтобы все нужные библиотеки, а также path были доступны
path = '/home/timof/.cache/kagglehub/datasets/kevinpatel04/celeba-original-wild-images/versions/1' # This 'path' is passed as images_path to the dataset
aligned_images_dir = 'PROJECT/FaceAlignment/aligned_images'

def create_heatmap(size, landmark, sigma=2):
    """
    Создаёт один heatmap с гауссовым ядром вокруг точки.

    :param size: (height, width) — размер heatmap'а
    :param landmark:(x, y) — координаты точки
    :param sigma
    :return: heatmap массив
    """
    x, y = landmark
    h, w = size

    # Обрезаем координаты, чтобы не выйти за пределы изображения
    x = min(max(0, int(x)), w - 1)
    y = min(max(0, int(y)), h - 1)

    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    heatmap = np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * sigma**2))
    return heatmap


def landmarks_to_heatmaps(image_shape, landmarks, sigma=2):
    """
    Преобразует список из N точек в набор из N heatmap'ов.

    :param image_shape: исходный размер изображения (H, W)
    :param landmarks: список из N пар координат [(x1, y1), (x2, y2), ..., (xN, yN),]
    :param sigma:
    :return: массив heatmap'ов вида [N, H, W]
    """
    heatmaps = []

    for (x, y) in landmarks:
        hm = create_heatmap(image_shape, landmark=(x,y), sigma=sigma)
        heatmaps.append(hm)

    return np.array(heatmaps)


# Функция для ресайза изображений с сохранением соотношений сторон
# Пустое пространство заполняется чёрным с помощью Pad
# При этом, она ещё и адаптирует положение лэндмарков, используя torchvision.tv_tensors.KeyPoints
class ResizeAndPad(T.Transform):
    def __init__(self, output_size=(128, 128)):
        super().__init__()
        assert isinstance(output_size, (int, tuple))
        # Смотрит, что пользователь указал в размерах изображения
        if isinstance(output_size, int):
            # Если он указал одно значение, то достраивает до квадарта
            self.output_size = (output_size, output_size)
        else:
            assert len(output_size) == 2
            self.output_size = output_size

    def forward(self, data):
        image = data['image'] # Содержит 'image' и 'keypoints'
        w, h = image.size
        target_w, target_h = self.output_size
        # Вычисляем коэффициент масштабирования, чтобы вписать изображение в target_size, сохраняя соотношение сторон
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)

        # Изменяем размер данных (изображение и ключевые точки). T.Resize автоматически обрабатывает tv_tensors.
        # Используем список для size=(new_h, new_w), так как T.Resize ожидает последовательность.
        data = T.Resize(size=(new_h, new_w), interpolation=T.InterpolationMode.BICUBIC)(data)

        # Пересчитываем размеры для заполнения после изменения размера, так как фактические new_w, new_h могут незначительно отличаться из-за преобразования в int()
        current_w, current_h = data['image'].size

        # Вычисляем размеры отступов для current_w, current_h, чтобы достичь target_w, target_h
        pad_w = target_w - current_w
        pad_h = target_h - current_h
        padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)
        # Применяем заполнение к данным. T.Pad автоматически обрабатывает tv_tensors.
        data = T.Pad(padding=padding, fill=0)(data)

        return data # Возвращает трансформированное изображение и адаптированные под него лэндмарки


# Функция для выравнивания изображений
# Она использует функцию face_align, которая будет написана ближе к концу ноутбука
class FaceAlign(T.Transform):
    def __init__(self, target_size=(128, 128), target_left_eye=(0.28, 0.35), target_right_eye=(0.72, 0.35), target_mouth_y=0.75, aligned_images_dir=None):
        super().__init__()
        assert isinstance(target_size, (int, tuple))
        if isinstance(target_size, int):
            self.target_size = (target_size, target_size)
        else:
            assert len(target_size) == 2
            self.target_size = target_size
        self.target_left_eye = target_left_eye
        self.target_right_eye = target_right_eye
        self.target_mouth_y = target_mouth_y
        self.aligned_images_dir = aligned_images_dir # Store the directory path

    def forward(self, data):
        image_pil = data['image']
        keypoints_obj = data['keypoints'] # Это объект KeyPoints
        image_id = data.get('image_id') # Получаем image_id из данных

        # Проверяем, существует ли уже выровненное изображение
        if self.aligned_images_dir and image_id:
            aligned_path = os.path.join(self.aligned_images_dir, image_id)
            if os.path.exists(aligned_path):
                # Если изображение существует, загружаем его и возвращаем
                aligned_image_pil = Image.open(aligned_path).convert('RGB')
                # Создаем пустой объект KeyPoints, так как для уже выровненного изображения они не нужны
                dummy_keypoints = KeyPoints(torch.empty(0, 2), canvas_size=self.target_size)
                return {'image': aligned_image_pil, 'keypoints': dummy_keypoints}


        # Если изображения нет в кэше, выполняем выравнивание
        # Конвертируем PIL Image в массив NumPy (H, W, C)
        image_np = np.array(image_pil)

        # Извлекаем ориентиры из объекта KeyPoints (конвертируем в массив NumPy)
        landmarks_np = keypoints_obj.data.numpy() # Форма (N, 2)

        # Вызываем функцию face_align
        # face_align ожидает image_np (H, W, C) и landmarks (N, 2)
        aligned_face_np, M = face_align(image_np,
                                        landmarks_np,
                                        target_size=self.target_size,
                                        target_left_eye=self.target_left_eye,
                                        target_right_eye=self.target_right_eye,
                                        target_mouth_y=self.target_mouth_y)

        # Конвертируем выровненный массив NumPy обратно в PIL Image
        aligned_image_pil = Image.fromarray(aligned_face_np)

        # Ориентиры нам нужны были только для выравнивания. Теперь они нам не нужны
        dummy_keypoints = KeyPoints(torch.empty(0, 2), canvas_size=self.target_size)

        # Возвращаем трансформированное изображение и ключевые точки
        return {'image': aligned_image_pil, 'keypoints': dummy_keypoints}


class FaceRecognitionDataset(Dataset):
    def __init__(self, df, images_path, aligned_images_dir, target_image_size=(128, 128), transform=None, augment_transform=False, mode='landmark_prediction', label_encoder=None):
        self.df = df
        self.images_path = images_path
        self.aligned_images_dir = aligned_images_dir
        self.target_image_size = target_image_size
        self.mode = mode
        self.label_encoder = label_encoder # Global label encoder for person_id

        if transform is None:
            # Определяем базовые преобразования, включая масштабирование и заполнение
            base_transforms = []

            # Если align_image = True, добавляем трансформацию FaceAlign
            if mode == 'face_recognition':
                # FaceAlign также обработает изменение размера до target_image_size внутри
                # Передаем aligned_images_dir в FaceAlign для возможности кеширования
                base_transforms.append(FaceAlign(target_size=target_image_size, aligned_images_dir=self.aligned_images_dir))
            elif mode == 'landmark_prediction':
                # В противном случае используем ResizeAndPad
                base_transforms.append(ResizeAndPad(target_image_size))
            else:
                raise ValueError(f"Неизвестный mode: {self.mode}. Выберите 'face_recognition' или 'landmark_prediction'.")


            # Добавляем аугментации, если augment_transform предоставлен, или используем набор по умолчанию
            if augment_transform == True:
                # Аугментации по умолчанию (для обучения распознавателя позже)
                augment_transforms = [
                    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
                    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                    T.RandomHorizontalFlip(p=0.5),
                ]
            else:
                augment_transforms = []

            # Объединяем аугментации с базовыми преобразованиями для конечного пайплайна
            self.transform = T.Compose(base_transforms + augment_transforms + [
                T.PILToTensor(),
                T.ConvertImageDtype(torch.float),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        else:
            # Если предоставлено пользовательское преобразование, используем его напрямую
            self.transform = transform


    def __len__(self):
        return len(self.df)

    def _get_image(self, image_id):
        # Получение изображения из исходной папки с каггла
        part = (int(image_id[:-4]) - 1) // 10000 + 1
        image_full_path = os.path.join(self.images_path, f'Part {part}', f'Part {part}', image_id)
        try:
            image = Image.open(image_full_path).convert('RGB')
            return image
        except FileNotFoundError:
            print(f'Путь не найден. Возможно файла "{image_id}" не существует. Пропускаем изображение.')
            return None


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        person_id = row['person_id']

        # Получаем изображение
        image = self._get_image(image_id)
        if image is None:
            return None

        # Получаем исходные размеры ббокса
        x1_orig, y1_orig = row['x_1'], row['y_1']
        cropped_width, cropped_height = row['width'], row['height']

        if cropped_width == 0 or cropped_height == 0:
            print(f"Предупреждение: Пропускаем {image_id} из-за недопустимых ориентиров (нулевая ширина/высота bbox).")
            return None

        # Обрезаем изображения по координатам ббокса
        x2_orig, y2_orig = x1_orig + cropped_width, y1_orig + cropped_height
        image = image.crop((x1_orig, y1_orig, x2_orig, y2_orig))

        # Извлекаем абсолютные лэндмарки из датафрейма
        original_landmarks_abs = [
            (row['lefteye_x'], row['lefteye_y']),
            (row['righteye_x'], row['righteye_y']),
            (row['nose_x'], row['nose_y']),
            (row['leftmouth_x'], row['leftmouth_y']),
            (row['rightmouth_x'], row['rightmouth_y'])
        ]

        # Преобразуем абсолютные лэндмарки в координаты относительно ббоксов
        relative_landmarks_coords = []
        for lx_abs, ly_abs in original_landmarks_abs:
            lx_relative = lx_abs - x1_orig
            ly_relative = ly_abs - y1_orig
            relative_landmarks_coords.append((lx_relative, ly_relative))

        # Создаем объект Keypoints с координатами относительно обрезанного изображения
        keypoints_v2 = KeyPoints(torch.tensor(relative_landmarks_coords, dtype=torch.float),
                                   canvas_size=(cropped_height, cropped_width))

        # Применяем пайплайн преобразований к словарю, содержащему изображение и ключевые точки
        transformed_data = self.transform({'image': image, 'keypoints': keypoints_v2, 'image_id': image_id})

        # Получаем трансформированные изображение и лэндмарки
        transformed_image = transformed_data['image']
        transformed_keypoints_obj = transformed_data['keypoints']


        if self.mode == 'face_recognition':
            if self.label_encoder is None:
                raise ValueError("В моде 'face_recognition' LabelEncoder должен быть предоставлен")
            encoded_person_id = self.label_encoder.transform([person_id])[0]
            return transformed_image, encoded_person_id
        elif self.mode == 'landmark_prediction':
            # This part remains as original for landmark_prediction mode
            final_landmarks_for_heatmap_and_plotting = transformed_keypoints_obj.data.cpu().numpy().tolist()

            # Определяем целевой размер хитмапы
            heatmap_target_h, heatmap_target_w = 64, 64
            input_image_h, input_image_w = self.target_image_size # (128, 128)

            # Масштабируем ориентиры из размера входного изображения (128x128) к целевому размеру хитмапы (64x64)
            scale_factor_h_for_heatmap = heatmap_target_h / input_image_h
            scale_factor_w_for_heatmap = heatmap_target_w / input_image_w

            scaled_landmarks_for_heatmap = []
            for lx, ly in final_landmarks_for_heatmap_and_plotting:
                scaled_landmarks_for_heatmap.append((int(round(lx * scale_factor_w_for_heatmap)),
                                                     int(round(ly * scale_factor_h_for_heatmap))))

            # Переводим лэндмарки в хитмапы
            heatmaps_np = landmarks_to_heatmaps((heatmap_target_h, heatmap_target_w), scaled_landmarks_for_heatmap)
            heatmaps_tensor = torch.from_numpy(heatmaps_np).float()

            # Также переводим лэндмарки в тензор
            adjusted_landmarks_tensor = torch.tensor(final_landmarks_for_heatmap_and_plotting).float()

            return transformed_image, heatmaps_tensor, adjusted_landmarks_tensor
        else:
            raise ValueError(f"Неизвестный mode: {self.mode}. Выберите 'face_recognition' или 'landmark_prediction'.")

# Function custom_collate_fn for filtering None-samples (remains same)
def custom_collate_fn(batch):
    # Filter None-samples (e.g., failed image loading)
    batch = [item for item in batch if item is not None]
    if not batch:
        # If batch is empty after filtering, return None to be skipped by DataLoader
        return None

    return torch.utils.data.dataloader.default_collate(batch)

In [2]:
import pandas as pd

train_dataset_df = pd.read_csv('PROJECT/FaceAlignment/train_dataset.csv')
val_dataset_df = pd.read_csv('PROJECT/FaceAlignment/val_dataset.csv')
test_dataset_df = pd.read_csv('PROJECT/FaceAlignment/test_dataset.csv')

In [3]:
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("kevinpatel04/celeba-original-wild-images")

landmarks = pd.read_csv('PROJECT/FaceAlignment/pred_landmarks.csv')
bboxes = pd.read_csv(f'{path}/list_bbox_celeba.csv')

# Helper function to merge a dataset dataframe with landmarks and bboxes
def merge_datasets(dataset_df, landmarks_df, bboxes_df):
    merged_df = pd.merge(dataset_df, landmarks_df, on='image_id', how='left')
    merged_df = pd.merge(merged_df, bboxes_df, on='image_id', how='left')
    return merged_df

# Merge train, val, and test dataframes with landmarks and bboxes
full_train_dataset_df = merge_datasets(train_dataset_df, landmarks, bboxes)
full_val_dataset_df = merge_datasets(val_dataset_df, landmarks, bboxes)
full_test_dataset_df = merge_datasets(test_dataset_df, landmarks, bboxes)

In [4]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

# Collect all unique person_ids across all dataframes
all_person_ids = pd.concat([
    full_train_dataset_df['person_id'],
    full_val_dataset_df['person_id'],
    full_test_dataset_df['person_id']
]).unique()

# Sort the person_ids to ensure consistent encoding
all_person_ids_sorted = np.sort(all_person_ids)

# Initialize and fit the LabelEncoder
global_label_encoder = LabelEncoder()
global_label_encoder.fit(all_person_ids_sorted)

# Update the global num_classes based on the fitted encoder
num_classes = len(global_label_encoder.classes_)

print(f"Updated num_classes: {num_classes}")

Updated num_classes: 2944


In [5]:
import torch
from torch.utils.data import DataLoader, default_collate

fixed_image_size = (128, 128) # Определяем фиксированный размер для всех изображений
# num_classes = full_train_dataset_df['person_id'].nunique() # REMOVED: num_classes is now globally defined from global_label_encoder

train_dataset = FaceRecognitionDataset(
    full_train_dataset_df,
    path,
    aligned_images_dir=aligned_images_dir,
    mode='face_recognition',
    label_encoder=global_label_encoder,
    target_image_size=fixed_image_size
    )

val_dataset = FaceRecognitionDataset(
    full_val_dataset_df,
    path,
    aligned_images_dir=aligned_images_dir,
    mode='face_recognition',
    label_encoder=global_label_encoder,
    target_image_size=fixed_image_size
    )

test_dataset = FaceRecognitionDataset(
    full_test_dataset_df,
    path,
    aligned_images_dir=aligned_images_dir,
    mode='face_recognition',
    label_encoder=global_label_encoder,
    target_image_size=fixed_image_size
    )

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn, num_workers=0)

In [6]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# num_classes will now be defined globally in a previous cell (XbsKIrcHEXxt)

class FaceRecognitionBackbone(nn.Module):
    def __init__(self, backbone_model=None, embedding_dim=512):
        super(FaceRecognitionBackbone, self).__init__()
        # Load a pre-trained ResNet50 model
        if backbone_model is None:
            self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        else:
            # If a custom backbone is provided, use it
            self.backbone = backbone_model

        if hasattr(self.backbone, 'fc'):
            self.backbone.fc = nn.Identity()
        elif hasattr(self.backbone, 'classifier'): # For models like EfficientNet
            self.backbone.classifier = nn.Identity()

        # Add other conditions here for different model architectures if needed

        # Dynamically infer in_features for embedding_head
        # Create a dummy input to pass through the backbone to get the output shape
        # Defaulting to 224x224, but this might need adjustment for specific models (e.g., EfficientNet-B7 expects 600x600)
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            # Pass the dummy input through the modified backbone (without the FC layer/classifier)
            # Flatten the output if it's not already 2D (e.g., coming from Conv layers)
            dummy_output = self.backbone(dummy_input)
            # If the output is (1, C, H, W), flatten to (1, C*H*W) before getting size(1)
            if dummy_output.dim() > 2:
                in_features = dummy_output.view(dummy_output.size(0), -1).size(1)
            else:
                in_features = dummy_output.size(1)

        # Add a new fully connected layer for embedding
        self.embedding_head = nn.Linear(in_features, embedding_dim)

        # Add a batch normalization layer after the embedding head
        self.bn = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        # Pass input through the ResNet backbone
        x = self.backbone(x)

        # If the output is still a feature map (e.g., from EfficientNet's features), flatten it
        if x.dim() > 2:
            x = x.view(x.size(0), -1)

        # Pass through the embedding head
        x = self.embedding_head(x)

        # Pass through batch normalization
        x = self.bn(x)

        return x

In [7]:
import math

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
        # Debug print outside forward to confirm init value
        # print(f"ArcFace initialized with out_features: {self.out_features}")

    def forward(self, input, label):
        # Normalize the input embeddings and weights
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m

        # Apply margin to target classes
        if self.m > 0:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Clamp labels to ensure they are within the valid range [0, num_classes-1]
        # This prevents F.one_hot from producing an empty tensor if labels are out of bounds.
        labels_clamped = torch.clamp(label, 0, self.out_features - 1)
        one_hot = F.one_hot(labels_clamped, num_classes=self.out_features).float()

        # Combine original logits with marginalized logits for target classes
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)

        # Scale the logits
        output *= self.s

        return output

In [8]:
import math
import torch.nn.functional as F

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
        print(f"ArcFace initialized with out_features: {self.out_features}")

    def forward(self, input, label):
        # Normalize the input embeddings and weights
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m

        # Apply margin to target classes
        if self.m > 0:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Clamp labels to ensure they are within the valid range [0, num_classes-1]
        # This prevents F.one_hot from producing an empty tensor if labels are out of bounds.
        labels_clamped = torch.clamp(label, 0, self.out_features - 1)

        one_hot = F.one_hot(labels_clamped, num_classes=self.out_features).float()
        # Combine original logits with marginalized logits for target classes
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)

        # Scale the logits
        output *= self.s

        return output

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class AdaCos(nn.Module):
    def __init__(self, in_features, out_features, m=0.50):
        super(AdaCos, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Initialize s dynamically during training, this will be handled in the forward pass.
        self.s = math.sqrt(2) * math.log(out_features - 1)

    def forward(self, input, label):
        # Normalize input embeddings and weights
        normalized_input = F.normalize(input)
        normalized_weight = F.normalize(self.weight)

        # Calculate cosine similarity
        cosine = F.linear(normalized_input, normalized_weight)

        # Apply margin to target classes
        if self.m > 0:
            theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
            marginal_theta = theta + self.m
            cosine_with_margin = torch.cos(marginal_theta)
        else:
            cosine_with_margin = cosine

        # Create one-hot labels for sparse target (label) and expand to match cosine shape
        one_hot = F.one_hot(label, num_classes=self.out_features).float()

        # Combine original logits with marginalized logits for target classes
        # For target classes, use cosine_with_margin, for others, use original cosine
        output = (one_hot * cosine_with_margin) + ((1.0 - one_hot) * cosine)

        # Adaptive scaling factor 's' (AdaCos specific)
        # This part requires specific implementation details of AdaCos, often an adaptive 's' based on cosine values
        # For simplicity, we can use a fixed 's' or adapt it based on a global mean of cosines if not directly implementing the full AdaCos paper's adaptive s.
        # The original AdaCos paper computes 's' based on the median of the norm of the feature vectors
        # and the median of the cosine similarity for the target class logits.
        # For this step, we will use a simplified adaptive scaling based on the median of current batch's target cosines.

        with torch.no_grad():
            B_avg = torch.where(one_hot == 1, cosine, torch.zeros_like(cosine))
            B_avg = B_avg.sum(dim=1) / one_hot.sum(dim=1)
            theta_median = torch.acos(torch.clamp(B_avg, -1.0 + 1e-7, 1.0 - 1e-7)).median()
            self.s = torch.log(self.weight.size(0) - 1.0) / torch.cos(theta_median * self.m) # Simplified adaptive s

        # Scale the logits
        output *= self.s

        return output

In [10]:
import torch.nn as nn

class LinearClassifier(nn.Module):
    def __init__(self, in_features, out_features):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, input):
        # For standard linear classification, we just pass the embeddings through a linear layer.
        # Normalization is not typically required here as it's often handled before this layer
        # or by the loss function itself (e.g., CrossEntropyLoss expects raw logits).
        logits = self.fc(input)
        return logits

In [11]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and LinearClassifier classes are available from previous steps

class FaceRecognitionModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=num_classes):
        super(FaceRecognitionModel, self).__init__()
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        self.classifier = LinearClassifier(in_features=embedding_dim, out_features=num_classes)

    def forward(self, x):
        embeddings = self.backbone(x)
        logits = self.classifier(embeddings)
        return logits

In [12]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and ArcFace classes are available from previous steps

class FaceRecognitionArcFaceModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=2944, s=64.0, m=0.50): # Explicitly set default num_classes
        super(FaceRecognitionArcFaceModel, self).__init__()
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        # Pass num_classes explicitly from the argument to ArcFace
        self.arcface = ArcFace(in_features=embedding_dim, out_features=num_classes, s=s, m=m)

    def forward(self, x, label):
        embeddings = self.backbone(x)
        # ArcFace layer expects both embeddings and the true labels
        logits = self.arcface(embeddings, label)
        return logits

In [13]:
import torch.nn as nn

# Ensure num_classes is available from previous steps (it was defined as 1944)
# Ensure FaceRecognitionBackbone and AdaCos classes are available from previous steps

class FaceRecognitionAdaCosModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=num_classes, m=0.50):
        super(FaceRecognitionAdaCosModel, self).__init__()
        self.backbone = FaceRecognitionBackbone(embedding_dim=embedding_dim)
        self.adacos = AdaCos(in_features=embedding_dim, out_features=num_classes, m=m)

    def forward(self, x, label):
        embeddings = self.backbone(x)
        # AdaCos layer expects both embeddings and the true labels
        logits = self.adacos(embeddings, label)
        return logits

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm.notebook import tqdm # Import tqdm

def train_model(model, train_dataloader, val_dataloader, optimizer, loss_criterion, num_epochs, display_freq, device):
    """
    Trains the given model.

    Args:
        model (torch.nn.Module): The neural network model to be trained.
        train_dataloader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        val_dataloader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        optimizer (torch.optim.Optimizer): The optimization algorithm (e.g., Adam, SGD).
        loss_criterion (torch.nn.Module): The loss function (e.g., CrossEntropyLoss).
        num_epochs (int): The total number of epochs for training.
        display_freq (int): How often (in epochs) to print training progress and validation results.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') on which to perform training.
    """
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    model.to(device)

    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        # Wrap train_dataloader with tqdm for a progress bar
        train_loop = tqdm(train_dataloader, leave=False, desc=f"Epoch {epoch+1}/{num_epochs} (Train)")
        for batch in train_loop: # Iterate over batch directly
            if batch is None:
                # This batch was entirely filtered out by custom_collate_fn
                continue

            inputs, labels = batch # Expecting (image, label) from FaceRecognitionDataset in 'face_recognition' mode
            inputs = inputs.to(device)
            labels = labels.to(device).long() # Ensure labels are LongTensor
            labels = labels.view(-1) # Ensure labels are 1D by flattening

            # Skip batch if labels are empty after processing (should ideally not happen if batch is not None, but as a safeguard)
            if labels.numel() == 0:
                continue

            optimizer.zero_grad()

            # Handle models with ArcFace/AdaCos specific forward pass
            if isinstance(model, FaceRecognitionArcFaceModel) or isinstance(model, FaceRecognitionAdaCosModel):
                logits = model(inputs, labels)
            else:
                logits = model(inputs)

            loss = loss_criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(logits, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Update tqdm progress bar with current loss
            train_loop.set_postfix(loss=running_loss/total_samples, acc=correct_predictions/total_samples)

        epoch_train_loss = running_loss / total_samples if total_samples > 0 else 0.0
        epoch_train_acc = correct_predictions / total_samples if total_samples > 0 else 0.0
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval()  # Set model to evaluation mode
        val_running_loss = 0.0
        val_correct_predictions = 0
        val_total_samples = 0

        with torch.no_grad():  # Disable gradient calculation during validation
            # Wrap val_dataloader with tqdm for a progress bar
            val_loop = tqdm(val_dataloader, leave=False, desc=f"Epoch {epoch+1}/{num_epochs} (Validation)")
            for batch in val_loop: # Iterate over batch directly
                if batch is None:
                    # This batch was entirely filtered out by custom_collate_fn
                    continue

                inputs, labels = batch # Expecting (image, label) from FaceRecognitionDataset in 'face_recognition' mode
                inputs = inputs.to(device)
                labels = labels.to(device).long() # Ensure labels are LongTensor
                labels = labels.view(-1) # Ensure labels are 1D by flattening

                # Skip batch if labels are empty after processing
                if labels.numel() == 0:
                    continue

                if isinstance(model, FaceRecognitionArcFaceModel) or isinstance(model, FaceRecognitionAdaCosModel):
                    logits = model(inputs, labels)
                else:
                    logits = model(inputs)

                loss = loss_criterion(logits, labels)
                val_running_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(logits, 1)
                val_correct_predictions += (predicted == labels).sum().item()
                val_total_samples += labels.size(0)

                # Update tqdm progress bar with current validation loss
                val_loop.set_postfix(loss=val_running_loss/val_total_samples, acc=val_correct_predictions/val_total_samples)

        epoch_val_loss = val_running_loss / val_total_samples if val_total_samples > 0 else 0.0
        epoch_val_acc = val_correct_predictions / val_total_samples if val_total_samples > 0 else 0.0
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        # Print progress
        if (epoch + 1) % display_freq == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]\n "
                  f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}\n "
                  f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}\n")

    return history

In [15]:
import torch.optim as optim

# Initialize the ArcFace model
# num_classes is already defined from previous steps (2944)
arcface_model = FaceRecognitionArcFaceModel(embedding_dim=512, num_classes=num_classes) # Pass num_classes explicitly

# Define optimizer and loss criterion
optimizer_arcface = optim.AdamW(arcface_model.parameters(), lr=0.001)
loss_criterion_arcface = nn.CrossEntropyLoss()

# Set training parameters
num_epochs = 10 # You can adjust this
display_freq = 1 # Display results every epoch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training ArcFace model on device: {device}")

# Run training
arcface_history = train_model(
    arcface_model,
    train_dataloader,
    val_dataloader,
    optimizer_arcface,
    loss_criterion_arcface,
    num_epochs,
    display_freq,
    device
)

print("ArcFace model training complete.")

ArcFace initialized with out_features: 2944
Training ArcFace model on device: cuda


Epoch 1/10 (Train):   0%|          | 0/2663 [00:00<?, ?it/s]

KeyboardInterrupt: 